# Manufacturing Quality — bắt đầu ở đây
**GIẢ LẬP seed 42**. Shift+Enter từng ô. Dự đoán: một đơn vị có ba lỗi thì COUNT(*) sau JOIN sẽ là bao nhiêu?

In [ ]:
from pathlib import Path
import sys
root=Path.cwd()
while not (root/'quality_lab').is_dir() and root!=root.parent: root=root.parent
if not (root/'quality_lab').is_dir(): raise RuntimeError('Mở bằng python run_quality.py --notebook tại thư mục dự án')
sys.path.insert(0,str(root))
from quality_lab.learning import read_sql,learning_db
db=learning_db()
print('Database chỉ đọc:',db)


## 1. Nguồn và số dòng
Phân biệt đơn vị, kiểm tra, lỗi và sửa.

In [ ]:
read_sql('SELECT * FROM metadata')

In [ ]:
read_sql('SELECT (SELECT COUNT(*) FROM units) AS units,(SELECT COUNT(*) FROM inspections) AS inspections,(SELECT COUNT(*) FROM inspection_defects) AS defect_records;')

## 2. JOIN và mẫu số
Mỗi dòng lỗi khác một đơn vị lỗi.

In [ ]:
read_sql('SELECT i.inspection_no,COUNT(*) AS defect_records,COUNT(DISTINCT i.unit_id) AS defective_units FROM inspections i JOIN inspection_defects d USING(inspection_id) GROUP BY i.inspection_no;')

In [ ]:
read_sql('SELECT inspection_no,COUNT(*) AS all_inspected FROM inspections GROUP BY inspection_no;')

## 3. Trạng thái tại mốc
Một dòng mỗi đơn vị, không nhìn kiểm tra lại sau mốc.

In [ ]:
from quality_lab.analysis import sql_tables,compare
tables=sql_tables(db)
tables['units'].head(10)

In [ ]:
tables['shifts']

## 4. SQL đối chiếu pandas
Số đếm chính xác, tỷ lệ sai số 1e-8; không xuất nếu sai.

In [ ]:
tables=compare(db)
from quality_lab.analysis import STATES
assert (tables['units'][STATES].sum(axis=1)==1).all()
assert tables['batches'].unit_count.sum()==len(tables['units'])
print('Một trạng thái/đơn vị và tổng lô khớp')

## 5. Biểu đồ và ba nhận xét
Không kiểm tra thì tỷ lệ trống. Heatmap đếm dòng lỗi. Ca chỉ so hình khi đủ 100 mẫu.

In [ ]:
from quality_lab.analyze import export
from IPython.display import display,Image,Markdown
export(db,out=root/'outputs/notebook')
for path in sorted((root/'outputs/notebook').glob('*.png')): display(Image(filename=str(path)))
display(Markdown((root/'outputs/notebook/report.md').read_text(encoding='utf-8')))

## Tự làm
Chọn lô tháng 1, quan sát hết tháng 3, giải thích vì sao giữ kiểm tra tháng 2. Viết ba nhận xét có mẫu số; không suy ca gây lỗi hoặc sửa chứng minh tác động nhân quả.